In [ ]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
spark.version

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

Question 2: Yellow November 2025 
Read the November 2025 Yellow into a Spark Dataframe. 
Repartition the Dataframe to 4 partitions and save it to parquet. What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)?

In [ ]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [ ]:
df_yellow.repartition(4) \
    .write.parquet('data/pq/yellow/2025/11', mode='overwrite')

In [ ]:
!ls -lh data/pq/yellow/2025/11/

Question 3: Count records
How many taxi trips were there on the 15th of November?
Consider only trips that started on the 15th of November

In [ ]:
from pyspark.sql import functions as F

In [ ]:
trips_15_nov = df_yellow.filter(F.to_date(df_yellow.tpep_pickup_datetime) == '2025-11-15').count()

In [ ]:
print(f"Taxi trips on the 15th of November: {trips_15_nov}")

Question 4: Longest trip
What is the length of the longest trip in the dataset in hours?

In [ ]:
df_duration = df_yellow.withColumn(
    'trip_duration_hours',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

In [ ]:
max_duration = df_duration.select(F.max('trip_duration_hours')).collect()[0][0]

print(f"The longest trip lasted: {max_duration:.2f} hours")

Question 6: Least frequent pickup location zone \
Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [ ]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [ ]:
df_yellow.createOrReplaceTempView('yellow_trips')
df_zones.createOrReplaceTempView('zones')

In [ ]:
least_frequent_zone = spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(*) as trip_count
    FROM 
        yellow_trips t
    JOIN 
        zones z ON t.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
    ORDER BY 
        trip_count ASC
    LIMIT 1
""")

In [ ]:
least_frequent_zone.show()